In [0]:
#==============================================================
# Gold Notebook Configuration
#==============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid

**Read Source Configuration**

In [0]:
#==============================================================
# Read Source Configuration
#==============================================================

source_config_df = spark.sql("""

SELECT *

FROM adb_databricks_project_dev.metadata.source_config

WHERE active_flag = 'Y'

ORDER BY source_id

""")

display(source_config_df)

# Convert metadata into Python list
source_config_list = [

    row.asDict()

    for row in source_config_df.collect()

]

print(f"Total Active Entities : {len(source_config_list)}")

**Read Silver Table**

In [0]:
#==============================================================
# Function Name : read_silver_table()
#
# Purpose :
# Reads Silver table dynamically.
#
# Called From :
# Gold Driver
#
# Layer :
# Gold
#==============================================================

def read_silver_table(config):

    silver_table = (

        f"{config['target_catalog']}."
        f"{config['silver_schema']}."
        f"{config['silver_table']}"

    )

    print(f"Reading : {silver_table}")

    df = spark.table(silver_table)

    print(f"Records : {df.count()}")

    return df

**## Join Silver Tables**

In [0]:
#==============================================================
# Function Name : join_silver_tables()
#
# Purpose :
# Dynamically joins Silver tables.
#
# Layer :
# Gold
#==============================================================

from pyspark.sql import functions as F

def join_silver_tables(
    silver_tables
):

    #------------------------------------------------------
    # Customer is the Master Table
    #------------------------------------------------------

    gold_df = silver_tables["Customer"]

    #------------------------------------------------------
    # Order
    #------------------------------------------------------

    if "Order" in silver_tables:

        gold_df = gold_df.join(

            silver_tables["Order"]

            .select(

                "CustomerID",
                "OrderID",
                "ProductName",
                "Quantity",
                "UnitPrice",
                "OrderDate"

            ),

            on="CustomerID",

            how="left"

        )

    #------------------------------------------------------
    # Address
    #------------------------------------------------------

    if "Address" in silver_tables:

        gold_df = gold_df.join(

            silver_tables["Address"]

            .select(

                "CustomerID",

                "AddressID",

                "AddressLine1",

                "AddressLine2",

                F.col("City").alias(
                    "AddressCity"
                ),

                F.col("State").alias(
                    "AddressState"
                ),

                F.col("Country").alias(
                    "AddressCountry"
                ),

                "PostalCode",

                "AddressType"

            ),

            on="CustomerID",

            how="left"

        )

    #------------------------------------------------------
    # Payment (Future)
    #------------------------------------------------------

    if "Payment" in silver_tables:

        gold_df = gold_df.join(

            silver_tables["Payment"],

            on="CustomerID",

            how="left"

        )

    #------------------------------------------------------
    # Shipment (Future)
    #------------------------------------------------------

    if "Shipment" in silver_tables:

        gold_df = gold_df.join(

            silver_tables["Shipment"],

            on="CustomerID",

            how="left"

        )

    #------------------------------------------------------
    # Amount
    #------------------------------------------------------

    if (
        "Quantity" in gold_df.columns
        and
        "UnitPrice" in gold_df.columns
    ):

        gold_df = gold_df.withColumn(

            "Amount",

            F.col("Quantity") *
            F.col("UnitPrice")

        )

    #------------------------------------------------------
    # Gold Audit Column
    #------------------------------------------------------

    gold_df = gold_df.withColumn(

        "gold_load_time",

        F.current_timestamp()

    )

    return gold_df

**Apply Business Transformations**

In [0]:
def apply_gold_transformations(df):

    return (

        df.withColumn(

            "gold_load_time",

            F.current_timestamp()

        )

    )

**Write Gold Table**

In [0]:
#==============================================================
# Function Name : write_gold_table()
#
# Purpose :
# Writes transformed data into Gold table.
#
# Called From :
# Gold Driver
#
# Layer :
# Gold
#==============================================================

def write_gold_table(
    config,
    df
):

    gold_table = (

        f"{config['target_catalog']}."
        f"{config['gold_schema']}."
        f"{config['gold_table']}"

    )

    (
        df.write

        .format("delta")

        .mode("overwrite")

        .option("overwriteSchema", "true")

        .saveAsTable(gold_table)

    )

    print(f"Gold Table Loaded : {gold_table}")

In [0]:
#==============================================================
# Function Name : escape_sql()
#==============================================================

def escape_sql(value):

    if value is None:
        return ""

    return str(value).replace(
        "'",
        "''"
    )

** Insert Gold Execution Log**

In [0]:
#==============================================================
# Function Name : insert_execution_log()
#
# Purpose :
# Inserts Gold notebook execution details into execution log.
#==============================================================

def insert_execution_log(
    config,
    source_start_time,
    status,
    records_read,
    records_loaded,
    error_message=None
):

    layer = "GOLD"

    file_format = "AUTO"

    spark.sql(f"""

    INSERT INTO
    {config['target_catalog']}.metadata.execution_log

    VALUES
    (

        '{execution_id}',

        '{escape_sql(config["entity_name"])}',

        '{layer}',

        'FILE',

        '{file_format}',

        '{source_start_time}',

        current_timestamp(),

        '{escape_sql(status)}',

        {records_read},

        {records_loaded},

        '{escape_sql(error_message)}'

    )

    """)

**Gold Driver**

In [0]:
#==============================================================
# Gold Driver
#==============================================================

from datetime import datetime
import uuid

execution_id = str(uuid.uuid4())

config = source_config_list[0]

source_start_time = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

status = "SUCCESS"

error_message = None

records_read = 0

records_loaded = 0

try:

    #==========================================================
    # Read All Silver Tables Dynamically
    #==========================================================

    silver_tables = {}

    for cfg in source_config_list:

        entity_name = cfg["entity_name"]

        silver_tables[
            entity_name
        ] = read_silver_table(
            cfg
        )

    #==========================================================
    # Count Records
    #==========================================================

    records_read = sum(

        df.count()

        for df in silver_tables.values()

    )

    #==========================================================
    # Join Tables
    #==========================================================

    gold_df = join_silver_tables(

        silver_tables

    )

    #==========================================================
    # Business Transformation
    #==========================================================

    gold_df = apply_gold_transformations(

        gold_df

    )

    #==========================================================
    # Count Loaded Records
    #==========================================================

    records_loaded = gold_df.count()

    #==========================================================
    # Write Gold Table
    #==========================================================

    write_gold_table(

        config,

        gold_df

    )

except Exception as e:

    status = "FAILED"

    error_message = str(e)

    print(error_message)

#==============================================================
# Insert Execution Log
#==============================================================

insert_execution_log(

    config=config,

    source_start_time=source_start_time,

    status=status,

    records_read=records_read,

    records_loaded=records_loaded,

    error_message=error_message

)

print("")

print("===================================")
print("Gold Notebook Completed Successfully")
print("===================================")

In [0]:
%sql

SELECT *
FROM adb_databricks_project_dev.gold.order_summary;